# Installing required packages

If you don't already have the following packages installed, uncomment the following lines and run the cell to install them. The required packages are: `pandas`, `numpy`, `seaborn`, `matplotlib`, `datasets`, `transformers`, `scikit-learn`, `tqdm`, and `torch`.

In [ ]:
# ! pip install pandas
# ! pip install numpy
# ! pip install seaborn
# ! pip install matplotlib
# ! pip install datasets
# ! pip install transformers
# ! pip install scikit-learn
# ! pip install tqdm
# ! pip install torch

First we need to load any necessary packages

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from datasets import load_dataset
from transformers import logging, pipeline

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline

import time
from tqdm.auto import tqdm
import warnings


# Model Fitting

We will load the first dataset, Jigsaw, for building and training the logistic regression model. For this project, the dataset needs to be split into 3 parts: a 10,000 observation evaluation dataset, a 10,000 observation threshold dataset for Toxic-BERT, and the remaining data is used to train the logistic regression.

In [ ]:
jigsaw = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge")

train_df = jigsaw['train'].to_pandas()

summary_table = pd.DataFrame({
    'count': [train_df['toxic'].sum()],
    'proportion': [train_df['toxic'].mean()]
})

summary_table

In [ ]:
train_df.head()

In [ ]:
full_df = train_df.copy()

# final held-out evaluation set
train_pool_df, final_eval_df = train_test_split(
    full_df,
    test_size=10000,
    stratify=full_df["toxic"],
    random_state=507
)

train_pool_df = train_pool_df.reset_index(drop=True)
final_eval_df = final_eval_df.reset_index(drop=True)

# separate threshold set for BERT
train_main_df, bert_thresh_df = train_test_split(
    train_pool_df,
    test_size=10000,
    stratify=train_pool_df["toxic"],
    random_state=508
)

train_main_df = train_main_df.reset_index(drop=True)
bert_thresh_df = bert_thresh_df.reset_index(drop=True)

Building the logistic regression model

In [ ]:
pipeline_logistic = Pipeline([
     ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 3))),
     ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced'))
 ])

This code chunk splits the training data in to five subsets for cross-validation training

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=507)

Assessing model fit

In [ ]:
scores = cross_val_score(
    pipeline_logistic,
    train_pool_df['comment_text'],
    train_pool_df['toxic'],
    cv=cv,
    scoring='roc_auc'   
)

print("AUC scores:", scores)
print("Mean AUC:", scores.mean())

Training the final logistic regression model on the full Jigsaw dataset (except for the evaluation subset)

In [ ]:
pipeline_logistic.fit(train_pool_df['comment_text'], train_pool_df['toxic'])

In [ ]:
feature_names = pipeline_logistic.named_steps['tfidf'].get_feature_names_out()

n_unigrams = sum(' ' not in f for f in feature_names)
n_bigrams = sum(f.count(' ') == 1 for f in feature_names)
n_trigrams = sum(f.count(' ') == 2 for f in feature_names)

print("Unigrams:", n_unigrams)
print("Bigrams:", n_bigrams)
print("Trigrams:", n_trigrams)

Threshold fine tuning is performed by identifying the best F1 threshold which balances recall and precision. Thresholds across a range from .01 to .99 were considered.

In [ ]:
y_true = train_pool_df['toxic']

y_probs_cv = cross_val_predict(
    pipeline_logistic,
    train_pool_df['comment_text'],
    y_true,
    cv=cv,
    method='predict_proba'
)[:, 1]

In [ ]:
thresholds = np.linspace(0.01, 0.99, 100)

f1_list = []

for t in thresholds:
    y_pred_t = (y_probs_cv >= t).astype(int)
    f1_list.append(f1_score(y_true, y_pred_t, zero_division=0))

In [ ]:

best_idx = np.argmax(f1_list)
best_threshold = thresholds[best_idx]

print("Best F1 threshold:", best_threshold)
print("Best F1:", f1_list[best_idx])

In [ ]:
plt.plot(thresholds, f1_list, label="F1 Score")
plt.axvline(best_threshold, linestyle='--', label=f'Best F1 ({best_threshold:.2f})')

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("F1 vs Threshold (Cross-Validated)")
plt.legend()

plt.show()

Now that we have trained the logistic regression, we want to load Toxic-BERT from HuggingFace as a comparison. We use the Hugging Face Transformers pipeline API to load the pre-trained Toxic-BERT model and its tokenizer for text classification. It creates a ready-to-use function that takes raw text as input and returns predicted class probabilities (toxicity scores) for each label.

In [ ]:
logging.set_verbosity_error()

bert_pipe = pipeline(
        "text-classification",
        model="unitary/toxic-bert",
        top_k = None
    )

Note: the Toxic BERT threshold is calculated from a subset of 10,000 of the training observations to greatly reduce computational load. This still provides a solid understanding of Toxic BERT's performance and classification behavior on the Jigsaw data. Batching is used in the code below and subsequent chunks to further reduce computational load by splitting the dataset into many smaller subsets.

The code currently uses a saved cache from GitHub to reduce computation time. To run the code directly, users can change "use_saved_results = True" to False.

In [ ]:
use_saved_results = True

github_url = "https://raw.githubusercontent.com/enoracat/Stats_507_Final_Project/main/bert_thresh_res.pkl"

if use_saved_results == True:
    bert_thresh_res = pd.read_pickle(github_url)

else:
    warnings.filterwarnings("ignore")

    bert_thresh_df_copy = bert_thresh_df.copy()
    bert_thresh_df_copy["text"] = bert_thresh_df_copy["comment_text"].fillna("").astype(str)

    def score_texts(texts, batch_size=8):
        results = []
        start = time.time()

        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]

            batch_results = bert_pipe(
                batch,
                batch_size=batch_size,
                truncation=True,
                max_length=512
            )

            results.extend(batch_results)

            if i % (batch_size * 20) == 0:
                print(f"Processed {i} / {len(texts)} in {time.time() - start:.1f} seconds")

        return results

    bert_thresh_df_results = score_texts(bert_thresh_df_copy["text"].tolist())

    bert_thresh_res_df = pd.DataFrame(
        [{f'bert_{item["label"]}': item["score"] for item in row} for row in bert_thresh_df_results]
    )

    bert_thresh_res = pd.concat([bert_thresh_df_copy.reset_index(drop=True), bert_thresh_res_df], axis=1)

    bert_thresh_res.to_pickle("bert_thresh_res.pkl")

In [ ]:
y_eval_true = bert_thresh_res["toxic"].values
bert_probs_thresh = bert_thresh_res["bert_toxic"].values

In [ ]:
thresholds = np.linspace(0.01, 0.99, 100)

f1_list = []

for t in thresholds:
    y_pred_t = (bert_probs_thresh >= t).astype(int)
    f1_list.append(f1_score(y_eval_true, y_pred_t, zero_division=0))


best_idx = np.argmax(f1_list)
best_threshold = thresholds[best_idx]

print("Best F1 threshold:", best_threshold)
print("Best F1:", f1_list[best_idx])

In [ ]:
plt.plot(thresholds, f1_list, label="F1 Score")
plt.axvline(best_threshold, linestyle='--', label=f'Best F1 ({best_threshold:.2f})')

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("F1 vs Threshold (Cross-Validated)")
plt.legend()

plt.show()

# Model Evaluation on Labeled Data

Now that we have the thresholds, I need to compare performance on the Jigsaw evaluation subset to determine how both models perform on labeled data. The table below directly compares several performance metrics for logistic regression and Toxic-BERT. The plot below shows differences in predicted toxicity probabilities.

The code currently uses a saved cache from GitHub to reduce computation time. To run the code directly, users can change "use_saved_results = True" to False.

In [ ]:
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import warnings

from transformers import logging, pipeline

use_saved_results = True

github_url = "https://raw.githubusercontent.com/enoracat/Stats_507_Final_Project/main/bert_eval_scores.pkl"

if use_saved_results == True:
    bert_eval = pd.read_pickle(github_url)

else:
    warnings.filterwarnings("ignore")

    final_eval_df_copy = final_eval_df.copy()
    final_eval_df_copy["text"] = final_eval_df_copy["comment_text"].fillna("").astype(str)

    def score_texts(texts, batch_size=8):
        results = []
        start = time.time()

        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]

            batch_results = bert_pipe(
                batch,
                batch_size=batch_size,
                truncation=True,
                max_length=512
            )

            results.extend(batch_results)

            if i % (batch_size * 20) == 0:
                print(f"Processed {i} / {len(texts)} in {time.time() - start:.1f} seconds")

        return results

    bert_eval_results = score_texts(final_eval_df_copy["text"].tolist())

    bert_score_df = pd.DataFrame(
        [{f'bert_{item["label"]}': item["score"] for item in row} for row in bert_eval_results]
    )

    bert_eval = pd.concat([final_eval_df_copy.reset_index(drop=True), bert_score_df], axis=1)

    bert_eval.to_pickle("bert_eval_scores.pkl")

y_eval = bert_eval["toxic"].values
bert_probs = bert_eval["bert_toxic"].values
bert_pred = (bert_probs >= 0.495).astype(int)


In [ ]:
X_eval = final_eval_df["comment_text"].fillna("").astype(str)
y_eval = final_eval_df["toxic"].values

logreg_probs = pipeline_logistic.predict_proba(X_eval)[:, 1]
logreg_pred = (logreg_probs >= .77).astype(int)

In [ ]:
results_compare = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "ROC_AUC": roc_auc_score(y_eval, logreg_probs),
        "Precision": precision_score(y_eval, logreg_pred),
        "Recall": recall_score(y_eval, logreg_pred),
        "F1": f1_score(y_eval, logreg_pred)
    },
    {
        "Model": "Toxic-BERT",
        "ROC_AUC": roc_auc_score(y_eval, bert_probs),
        "Precision": precision_score(y_eval, bert_pred),
        "Recall": recall_score(y_eval, bert_pred),
        "F1": f1_score(y_eval, bert_pred)
    }
])

results_compare

In [ ]:
eval_compare = pd.DataFrame({
    "true": y_eval,
    "logreg_prob": logreg_probs,
    "logreg_pred": logreg_pred,
    "bert_prob": bert_probs,
    "bert_pred": bert_pred
})


(eval_compare["logreg_pred"] != eval_compare["bert_pred"]).mean()

In [ ]:
sns.kdeplot(eval_compare["logreg_prob"], label="Logistic Regression", fill=True, color = "r")
sns.kdeplot(eval_compare["bert_prob"], label="Toxic-BERT", fill=True, color = "steelblue")

plt.xlabel("Predicted Probability")
plt.title("Smoothed Distribution of Predicted Toxicity Scores on the Jigsaw Evaluation Subset")
plt.legend()
plt.show()

# Reddit Data Analysis

Before beginning analysis, we need to load the data. The dataset was originally obtained from HuggingFace: https://huggingface.co/datasets/mo-mittal/reddit_political_subs and contains 7,503 posts from several political subreddits.

However, it requires an old version of the datasets package to access directly via HuggingFace, so I have uploaded it to Github to avoid package version issues when accessing the data.

In [ ]:
url = "https://raw.githubusercontent.com/enoracat/Stats_507_Final_Project/main/reddit_political_subs_dataset.csv"
df = pd.read_csv(url)

df.head()

As shown in the plot below, posts are evenly distributed from 9 subreddits

In [ ]:
fig1 = sns.countplot(data=df, x = 'subreddit')
fig1.set(xlabel='Subreddit Name', ylabel = 'Count', title='Frequency of Posts Across Subreddits')
plt.xticks(rotation=45, ha='right')
plt.show()

I will start by evaluating the Reddit data with logistic regression

In [ ]:
titles = df['title'].fillna('').astype(str)
probs = pipeline_logistic.predict_proba(titles)[:, 1]

df['logreg_prob'] = probs
df['logreg_pred'] = (probs >= 0.77).astype(int)

desc_stats_logreg = df['logreg_prob'].describe()
n_toxic_logreg = df['logreg_pred'].sum()
prop_toxic_logreg = df['logreg_pred'].mean()

print("Logistic Regression Probability and Classification Summary")
print(desc_stats_logreg.to_frame(name="logred_prob").round(4))
print("Number labeled toxic:", n_toxic_logreg)
print("Proportion labeled toxic:", prop_toxic_logreg)

In [ ]:
df['logreg_pred_oldthresh'] = (probs >= 0.5).astype(int)

n_toxic_logreg_oldthresh = df['logreg_pred_oldthresh'].sum()
prop_toxic_logreg_oldthresh = df['logreg_pred_oldthresh'].mean()

print("Logistic Regression Classification Summary at Old Threshold (0.5)")
print("Number labeled toxic:", n_toxic_logreg_oldthresh)
print("Proportion labeled toxic:", prop_toxic_logreg_oldthresh)

Now evaluating the Reddit data with Toxic-BERT

The code currently uses a saved cache from GitHub to reduce computation time. To run the code directly, users can change "use_saved_results = True" to False.

In [ ]:
use_saved_results = True

github_url = "https://raw.githubusercontent.com/enoracat/Stats_507_Final_Project/main/bert_toxicity_df.pkl"

if use_saved_results == True:
    bert_reddit = pd.read_pickle(github_url)
    
else:
    warnings.filterwarnings("ignore")

    df = df.copy()
    df["text"] = df["title"].fillna("").astype(str)

    def score_texts(texts, batch_size=8):
        results = []
        start = time.time()

        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            batch_results = bert_pipe(batch, batch_size=batch_size, truncation=True, max_length=512)
            results.extend(batch_results)

            if i % (batch_size * 20) == 0:
                print(f"Processed {i} / {len(texts)} in {time.time() - start:.1f} seconds")

        return results

    reddit_bert_toxicity = score_texts(df["text"].tolist())

    bert_score_df = pd.DataFrame(
        [{item["label"]: item["score"] for item in row} for row in reddit_bert_toxicity]
    )


    bert_toxicity_df = pd.concat([df.reset_index(drop=True), bert_score_df], axis=1)
    
    bert_toxicity_df.to_pickle("bert_toxicity_df.pkl")

    bert_reddit = bert_toxicity_df

df['bert_prob'] = bert_reddit['toxic']
df['bert_pred'] = (bert_reddit['toxic'] >= 0.495).astype(int)

desc_stats_bert = df['bert_prob'].describe()
n_toxic_bert = df['bert_pred'].sum()
prop_toxic_bert = df['bert_pred'].mean()

print("Toxic-BERT Probability and Classification Summary")
print(desc_stats_bert.to_frame(name="bert_prob").round(4))
print("Number labeled toxic:", n_toxic_bert)
print("Proportion labeled toxic:", prop_toxic_bert)

The plot below shows differences in the distributions of predicted probabilities

In [ ]:
plt.figure(figsize=(7, 4))

sns.kdeplot(df["logreg_prob"], label="Logistic Regression", fill=True, color = "r")
sns.kdeplot(df["bert_prob"], label="Toxic-BERT", fill=True, color = "steelblue")

plt.xlabel("Predicted Probability")
plt.title("Smoothed Distribution of Predicted Toxicity Scores")
plt.legend()
plt.show()

Assessing disagreement in classification between logistic regression and Toxic-BERT

In [ ]:
pd.crosstab(
    df['logreg_pred'],
    df['bert_pred'],
    rownames=['LogReg'],
    colnames=['BERT']
)

To understand why the models are creating different predictions, it is useful to look at some of the observations which they disagreed on

In [ ]:
pd.set_option('display.max_colwidth', None)
df.loc[df['logreg_pred'] != df['bert_pred'], ['title', 'logreg_prob', 'bert_prob']].head(30)

We can also visualize the disagreement at the level of individual posts

In [ ]:
df_sample = df.sample(n=2000, random_state=1)

plt.scatter(
    df_sample["logreg_prob"],
    df_sample["bert_prob"],
    alpha=0.3,
    s = 20
)

plt.axvline(0.77, color="black", linestyle="--", linewidth=1, alpha = 0.8)
plt.axhline(0.495, color="blacK", linestyle="--", linewidth=1, alpha = 0.8)

plt.xlabel("Logistic Regression Probability")
plt.ylabel("Toxic-BERT Probability")
plt.title("Model Disagreement in Toxicity Predictions")

plt.xlim(0, 1)
plt.ylim(0, 1)

plt.tight_layout()
plt.show()

After considering disagreement overall, it is also useful to consider if disagreement varies by specific post features: 1. Subreddit, 2. post length, and 3. post score

First looking at subreddit

In [ ]:
df['disagree'] = (df['logreg_pred'] != df['bert_pred']).astype(int)

subreddit_toxic = df.groupby('subreddit').agg(
    logreg_toxic_rate=('logreg_pred', 'mean'),
    bert_toxic_rate=('bert_pred', 'mean'),
    disagreement_rate=('disagree', 'mean'),
    n_posts=('title', 'size')
).reset_index()

subreddit_toxic['toxicity_diff'] = (
    subreddit_toxic['logreg_toxic_rate'] - subreddit_toxic['bert_toxic_rate']
)

print("Subreddit Toxicity Comparison")
print(
    subreddit_toxic
    .round(4)
    .to_string(index=False)
)

In [ ]:
subs = subreddit_toxic['subreddit']
x = np.arange(len(subs))

plt.figure(figsize=(7, 5))

plt.bar(x - 0.2, subreddit_toxic['logreg_toxic_rate'], width=0.4,
        label='Logistic Regression', color='#d8031c', edgecolor='black', linewidth=0.5)
plt.bar(x + 0.2, subreddit_toxic['bert_toxic_rate'], width=0.4,
        label='Toxic-BERT', color='#0371d8', edgecolor='black', linewidth=0.5)

plt.xticks(x, subs, rotation=45, ha='right')
plt.ylabel('Toxicity Rate (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.title('Toxicity Rate by Subreddit and Model')
plt.legend()

plt.tight_layout()
plt.savefig('subreddit_toxic.png', dpi=300, bbox_inches='tight')
plt.show()

Now moving to post length

In [ ]:
df['word_count'] = df['title'].str.split().str.len()
df['prob_diff'] = df['bert_prob'] - df['logreg_prob']

In [ ]:
df['length_bin'] = pd.qcut(df['word_count'], 60, duplicates='drop')

score_summary = df.groupby('length_bin', observed=True)['prob_diff'].mean()

plt.figure(figsize=(6,4))
plt.plot(range(len(score_summary)), score_summary.values, marker='o')
plt.xlabel('Word Count')
plt.ylabel('Average Probability Difference')
plt.title('Probability Difference vs Word Count')
plt.show()

In [ ]:
df['length_bin'] = pd.qcut(df['word_count'], q=10, duplicates='drop')

length_plot = df.groupby('length_bin', observed=False).agg(
    logreg_toxic_rate=('logreg_pred', 'mean'),
    bert_toxic_rate=('bert_pred', 'mean')
).reset_index()

length_plot['bin_label'] = length_plot['length_bin'].apply(
    lambda x: f"{int(x.left)}–{int(x.right)}"
)

plt.plot(length_plot['bin_label'], length_plot['logreg_toxic_rate'],
         marker='o', label='Logistic Regression', color='#d8031c')

plt.plot(length_plot['bin_label'], length_plot['bert_toxic_rate'],
         marker='o', label='Toxic-BERT', color='#0371d8')

plt.xticks(rotation=45, ha='right')
plt.ylabel('Toxicity Rate (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.xlabel('Word Count')
plt.title('Toxicity Rate vs Post Length (Binned)')
plt.legend()

plt.tight_layout()
plt.show()

Finally looking at post score

In [ ]:
df['score_bin'] = pd.qcut(df['score'], 20, duplicates='drop')

score_summary = df.groupby('score_bin', observed=True)['prob_diff'].mean()

plt.figure(figsize=(6,4))
plt.plot(range(len(score_summary)), score_summary.values, marker='o')
plt.xlabel('Score (Quantile Bins)')
plt.ylabel('Average Probability Difference')
plt.title('Probability Difference vs Score (Binned)')
plt.show()

In [ ]:
df['score_bin'] = pd.qcut(df['score'], q=10, duplicates='drop')

score_plot = df.groupby('score_bin', observed=False).agg(
    logreg_toxic_rate=('logreg_pred', 'mean'),
    bert_toxic_rate=('bert_pred', 'mean')
).reset_index()

score_plot['bin_label'] = score_plot['score_bin'].apply(
    lambda x: f"{int(x.left)}–{int(x.right)}"
)

plt.plot(score_plot['bin_label'], score_plot['logreg_toxic_rate'],
         marker='o', label='Logistic Regression', color='#d8031c')

plt.plot(score_plot['bin_label'], score_plot['bert_toxic_rate'],
         marker='o', label='Toxic-BERT', color='#0371d8')

plt.xticks(rotation=45, ha='right')
plt.ylabel('Toxicity Rate (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.xlabel('Score')
plt.title('Toxicity Rate vs Score (Binned)')
plt.legend()

plt.tight_layout()
plt.show()